# RAGAS scores
Start the backend (`cd backend; uvicorn main:app`), then **Run All**.
- Cell 2 reads the questions and correct answers from `ragas_dataset_small.jsonl` and asks `/chat/query` for each answer and the chunks it used. Answers are cached in `ragas_dataset_small_answered.jsonl`; delete that file to ask again.
- Chunk text comes from the response's `evaluation` field, so keep `INCLUDE_EVAL_DATA=true`.
- The judge is Gemini (`GEMINI_API_KEY` in `.env`), so questions, answers and chunk text are sent to Google.

In [1]:
import os
import warnings

os.environ["RAGAS_DO_NOT_TRACK"] = "true"  # must be set before ragas is imported
warnings.filterwarnings("ignore", category=DeprecationWarning)

from dotenv import find_dotenv, load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import answer_correctness, answer_relevancy, context_precision, context_recall, faithfulness

load_dotenv(find_dotenv(usecwd=True))  # repo-root .env, found by searching up from the notebook folder
assert os.getenv("GEMINI_API_KEY"), "Set GEMINI_API_KEY in the repo-root .env"

JUDGE_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")  # same Gemini model the backend uses
# bypass_n / bypass_temperature: Gemini rejects multiple candidates and fixes its own sampling.
llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model=JUDGE_MODEL), bypass_n=True, bypass_temperature=True)
embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="gemini-embedding-001"))
# Few parallel calls plus retries keep free-tier rate limits (429) from turning scores into NaN.
run_config = RunConfig(max_workers=2, timeout=300, max_retries=10, max_wait=90)

ModuleNotFoundError: No module named 'langchain_google_genai'

In [ ]:
import json
from pathlib import Path

import httpx

DATASET = Path("ragas_dataset_small.jsonl")  # swap for ragas_dataset.jsonl to run all 30
LIMIT = 1  # how many questions to run; None = all of them
ANSWERED = DATASET.with_name(DATASET.stem + "_answered.jsonl")  # cache; delete it to re-ask the chat
BACKEND_URL = "http://localhost:8000"

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

rows = read_jsonl(DATASET)[:LIMIT]
cache = {r["id"]: r for r in read_jsonl(ANSWERED)} if ANSWERED.exists() else {}
for row in rows:
    if row["id"] not in cache:  # only ask the chat about questions it hasn't answered yet
        reply = httpx.post(f"{BACKEND_URL}/chat/query", json={"query": row["user_input"]}, timeout=600)
        reply.raise_for_status()
        data = reply.json()
        row["response"] = data["response"]
        # Full text of every chunk the LLM was shown, in [n] marker order.
        row["retrieved_contexts"] = [item["chunk_text"] for item in data["evaluation"]["retrieved"]]
        cache[row["id"]] = row
        print(f"{row['id']}: {len(row['retrieved_contexts'])} chunks")
    row.update(cache[row["id"]])
ANSWERED.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in cache.values()), encoding="utf-8")

samples = [{k: row[k] for k in ("user_input", "response", "retrieved_contexts", "reference")} for row in rows]
len(samples)

In [ ]:
result = evaluate(
    EvaluationDataset.from_list(samples),
    metrics=[answer_correctness, answer_relevancy, faithfulness, context_precision, context_recall],
    llm=llm,
    embeddings=embeddings,
    run_config=run_config,
)
scores = ["answer_correctness", "answer_relevancy", "faithfulness", "context_precision", "context_recall"]
df = result.to_pandas()
df.insert(0, "id", [r["id"] for r in rows])
df.insert(1, "type", [r["type"] for r in rows])
display(df[["id", "type", "user_input", *scores]])
display(df.groupby("type")[scores].mean())
df[scores].mean()